# 03 — Silver Customers CDC

## Overview
This lab demonstrates **Change Data Capture (CDC) processing** — consuming a stream of database change events (INSERT, UPDATE, DELETE) from a Bronze CDC table and applying them idempotently to a Silver customer table using Delta MERGE.

**What you will learn**
- How to use a persistent **sequence-number watermark** as a cursor so pipelines are replay-safe
- How to resolve multiple CDC events for the same key within a single batch (last-write-wins)
- How to implement **soft deletes** — marking rows `is_deleted=true` instead of physically removing them, preserving history for SCD2 and auditing
- How to write separate MERGEs for upserts and deletes to keep the logic readable and correct
- How to advance the watermark atomically after a successful write

**Prerequisites:** Scenario 01 has been run; `bronze.customers_cdc` and `bronze.pipeline_watermark` tables exist.

### Cell 1 — Configuration & table references
Sets up the three table references used throughout. `pipeline_watermark` is a **persistent control table** that stores the highest `change_seq` value processed in the previous run. It acts as a durable cursor: if the pipeline crashes mid-run, the watermark is not updated, so the next run safely re-processes the same batch from scratch.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

LH           = "lh_advanced_scenarios"
CDC_TABLE    = f"{LH}.bronze.customers_cdc"
SILVER_TABLE = f"{LH}.silver.customers"
WATERMARK    = f"{LH}.bronze.pipeline_watermark"

### Cell 2 — Read watermark
Reads the last successfully processed `change_seq` from the control table. `COALESCE(..., -1)` handles the first-ever run when no watermark row exists yet. This value becomes the **exclusive lower bound** for the next read: `change_seq > last_seq` means "give me only new changes".

In [ ]:
# ── Read last processed change_seq ───────────────────────────────────────────
try:
    last_seq = spark.sql(
        f"SELECT COALESCE(MAX(CAST(last_value AS BIGINT)), -1) AS seq "
        f"FROM {WATERMARK} WHERE entity_name = 'customers_cdc'"
    ).collect()[0]["seq"]
except Exception:
    last_seq = -1

print(f"Last processed change_seq: {last_seq}")

### Cell 3 — Read new CDC rows
Fetches only CDC rows with `change_seq > last_seq`. The early exit via `dbutils.notebook.exit()` stops execution cleanly when there is nothing to process — this keeps Data Factory pipeline run logs uncluttered (a clean exit is not a failure) and avoids running expensive MERGE operations against empty datasets.

In [ ]:
# ── Read unprocessed CDC rows ────────────────────────────────────────────────
df_cdc = (
    spark.read.format("delta").table(CDC_TABLE)
         .filter(F.col("change_seq") > last_seq)
)
batch_count = df_cdc.count()
print(f"CDC rows to process: {batch_count:,}")
if batch_count == 0:
    dbutils.notebook.exit("No new CDC rows. Nothing to do.")

### Cell 4 — Resolve final state per customer (late-arrival protection)
A single CDC batch can contain multiple events for the same customer (e.g. an `UPDATE` followed by another `UPDATE` followed by a `DELETE`). The window function `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY change_seq DESC)` keeps only the **final state** — the event with the highest sequence number. Applying all events individually would risk applying them out of order; this resolution ensures last-write-wins semantics within the batch. PII (`email`) is also hashed here before touching Silver.

In [ ]:
# ── Resolve last op per customer in this batch (late-arrival protection) ─────
w = Window.partitionBy("customer_id").orderBy(F.col("change_seq").desc())
df_resolved = (
    df_cdc
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    # Hash PII
    .withColumn("email_hash", F.sha2(F.col("email"), 256))
    .drop("email")
    .withColumn("_silver_ts", F.current_timestamp())
)

max_seq = df_cdc.agg(F.max("change_seq")).collect()[0][0]
print(f"Max change_seq in batch: {max_seq}")

### Cell 5 — Create table or MERGE upserts & soft-deletes
**First run:** seeds the Silver table from the resolved CDC rows, skipping any DELETE events (nothing to delete yet). **Subsequent runs:** two separate MERGEs run in sequence:
1. **Upsert MERGE** (I/U ops) — updates existing customers only if `change_seq` is higher (prevents stale events from overwriting newer state), inserts new customers.
2. **Soft-delete MERGE** (D ops) — sets `is_deleted=true` on the matched row. Physical deletion is avoided so that the SCD2 notebook (Scenario 04) can see the deletion event and create a final expired Gold row.

The `change_seq > t.change_seq` guard on both MERGEs is the **idempotency key**: re-running with the same batch produces no net change.

In [ ]:
# ── Create Silver table if it does not exist ─────────────────────────────────
if not spark.catalog.tableExists(SILVER_TABLE):
    seed = df_resolved.filter(F.col("op_flag") != "D") \
                      .drop("op_flag") \
                      .withColumn("is_deleted", F.lit(False))
    seed.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(SILVER_TABLE)
    print(f"Created {SILVER_TABLE} (first load).")
else:
    silver = DeltaTable.forName(spark, SILVER_TABLE)

    # ── MERGE: upserts ────────────────────────────────────────────────────────
    df_upserts = df_resolved.filter(F.col("op_flag").isin(["I", "U"]))
    if df_upserts.count() > 0:
        (
            silver.alias("t")
                  .merge(df_upserts.alias("s"), "t.customer_id = s.customer_id")
                  .whenMatchedUpdate(
                      condition="s.change_seq > t.change_seq",
                      set={
                          "name":        "s.name",
                          "email_hash":  "s.email_hash",
                          "region":      "s.region",
                          "change_seq":  "s.change_seq",
                          "change_ts":   "s.change_ts",
                          "is_deleted":  "false",
                          "_silver_ts":  "s._silver_ts",
                      })
                  .whenNotMatchedInsert(values={
                      "customer_id": "s.customer_id",
                      "name":        "s.name",
                      "email_hash":  "s.email_hash",
                      "region":      "s.region",
                      "change_seq":  "s.change_seq",
                      "change_ts":   "s.change_ts",
                      "is_deleted":  "false",
                      "_silver_ts":  "s._silver_ts",
                  })
                  .execute()
        )

    # ── MERGE: soft deletes ───────────────────────────────────────────────────
    df_deletes = df_resolved.filter(F.col("op_flag") == "D")
    if df_deletes.count() > 0:
        (
            silver.alias("t")
                  .merge(df_deletes.alias("s"), "t.customer_id = s.customer_id")
                  .whenMatchedUpdate(
                      condition="s.change_seq > t.change_seq",
                      set={
                          "is_deleted": "true",
                          "change_seq": "s.change_seq",
                          "change_ts":  "s.change_ts",
                          "_silver_ts": "s._silver_ts",
                      })
                  .execute()
        )

    print("MERGE complete.")

### Cell 6 — Advance watermark
Updates the `pipeline_watermark` control table to `max_seq` of the just-processed batch. This MERGE is **transactionally isolated from the Silver write** — if the Silver MERGE succeeded but this step fails, the next run will re-process the same batch (applying the same idempotent changes again, safely). The watermark is never advanced before the data write succeeds.

In [ ]:
# ── Update watermark ─────────────────────────────────────────────────────────
spark.sql(f"""
    MERGE INTO {WATERMARK} AS t
    USING (SELECT 'customers_cdc' AS entity_name, 'change_seq' AS watermark_col,
                   CAST({max_seq} AS TIMESTAMP) AS last_value,
                   CURRENT_TIMESTAMP() AS updated_at) AS s
    ON t.entity_name = s.entity_name
    WHEN MATCHED THEN UPDATE SET t.last_value = s.last_value, t.updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"Watermark updated to change_seq={max_seq}")

### Cell 7 — Validation
Asserts uniqueness of `customer_id` in Silver (there must be exactly one row per customer — CDC MERGE should never produce duplicates) and prints the count of soft-deleted rows as an operational indicator. An unexpectedly high deleted count may signal a bulk-delete event in the source system worth investigating.

In [ ]:
# ── Validation ───────────────────────────────────────────────────────────────
dups = spark.sql(
    f"SELECT customer_id, COUNT(*) c FROM {SILVER_TABLE} GROUP BY customer_id HAVING c>1"
).count()
assert dups == 0, f"{dups} duplicate customer_ids found"

deleted = spark.sql(
    f"SELECT COUNT(*) c FROM {SILVER_TABLE} WHERE is_deleted = true"
).collect()[0][0]

total = spark.sql(f"SELECT COUNT(*) c FROM {SILVER_TABLE}").collect()[0][0]
print(f"Total: {total:,} | Deleted: {deleted} | Duplicates: {dups}")